# Session 16 · Exercises

Work in pairs. `help()` and `dir()` in Python, the lecture notebooks and the workspace code answer most questions. The lecturer chooses which exercises to set in the 30 minutes of each block; the others are good practice at home. *Paper* and *Predict* tasks are answered in the markdown cell below the task, *Code* tasks in the empty code cells. Stretch tasks are for pairs that finish early.

Run the setup cell first: it makes the paths of the case-study data work wherever Jupyter was started.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Part 1 · Web APIs and a released service

### 1.1 · Tomorrow's weather in JSON

*Paper · 5 min*

> The printout below is a shortened answer of a weather API for Berlin. Write the Python expression that returns tomorrow's maximum temperature, and one that finds the day with the most rain.

```json
{
  "latitude": 52.52, "longitude": 13.41, "timezone": "Europe/Berlin",
  "daily_units": {"time": "iso8601", "temperature_2m_max": "°C", "precipitation_sum": "mm"},
  "daily": {
    "time": ["2026-10-12", "2026-10-13", "2026-10-14", "2026-10-15"],
    "temperature_2m_max": [16.2, 14.8, 12.1, 13.5],
    "precipitation_sum": [0.0, 3.4, 7.9, 0.2]
  }
}
```

The answer has been parsed with `data = json.loads(text)`; today is the first date.

*Source:* original exercise for this course; background: [Open-Meteo API documentation](https://open-meteo.com/en/docs)

*Your answer:*



### 1.2 · API arithmetic

*Paper · 10 min*

> Your team needs the daily weather (three variables) of Berlin, Hamburg and Munich from 1 March 2022 to 31 August 2025. The client pages by calendar year, one request per city and year. Open-Meteo counts a request for more than 14 days as several calls: one request costs *days / 14* calls (at least 1). The free API allows 600 calls per minute, 5,000 per hour and 10,000 per day.
>
> 1. Which requests does the client send for one city (start and end date of each)? How many requests in total for the three cities?
> 2. How many calls does Open-Meteo count for the whole download?
> 3. A teammate puts the download at the top of the analysis notebook, which the team runs about 50 times a day. Which limit is reached first, and what should the notebook do instead?

*Source:* original exercise for this course; background: [Open-Meteo terms and pricing](https://open-meteo.com/en/pricing)

*Your answer:*



### 1.3 · Capacity in your head

*Paper · 5 min*

> The tariff service receives 100 requests per second, and one prediction takes 50 ms. How many workers (processes that each answer one request at a time) does the service need?

*Source:* [Little's law, Wikipedia](https://en.wikipedia.org/wiki/Little%27s_law)

*Your answer:*



### 1.4 · "Works on my machine"

*Pairs · 10 min*

> List five reasons why a notebook that runs on your laptop fails on someone else's computer. For each reason, name what in the workspace of this session prevents it.

*Source:* original exercise for this course; background: [The Twelve-Factor App](https://12factor.net/) (factors II Dependencies and III Config)

*Your answer:*



### 1.5 · The pickle mismatch

*Predict, then Code · 10 min*

> You save a fitted pipeline with `joblib.dump` using one version of scikit-learn and load it with another version. What happens? Write down your prediction first. Then try it and compare: save the pipeline in this notebook, and load it in a terminal with an older version, for example `uv run --no-project --with "scikit-learn==1.7.*" --with joblib python -c "import joblib; print(joblib.load('pipe.joblib'))"`.

*Source:* [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)

*Your answer:*



### 1.6 · An endpoint for heading texts

*Code · 15 min*

> Customs officers want to look up the English text of a heading. Write a small FastAPI app with one endpoint, `GET /headings/{heading}`, that returns `{"heading": "6403", "text": "..."}` from the dictionary `HEADINGS` loaded below. A heading that does not exist (for example `0100`) is answered with 404, and a path that is not four digits (for example `64x3` or `640`) with 422. Then check all three cases with `TestClient`, without starting a server.

Hint: `help(fastapi.Path)` shows how a path parameter gets a `pattern`.

In [ ]:
import pandas as pd

nomenclature = pd.read_parquet("case-study/data/nomenclature.parquet")
HEADINGS = dict(zip(nomenclature["heading"], nomenclature["heading_description"]))
print(len(HEADINGS), "headings, for example", "6403:", HEADINGS["6403"])

### 1.7 · A server that fails (*Stretch*)

*Code · 15 min*

> The `WeatherClient` in [weather.py](weather.py) retries after HTTP 429, a server error (5xx) or a timeout. Test with a mock server instead of Open-Meteo what it does when the server answers `503` twice and then returns one valid day. Use an `httpx.MockTransport` as the server, and pass the client a `sleep` function that records the waits instead of waiting. How many requests does the client send, and how long would it have waited in total? What happens if the server answers `503` five times in a row?

Hint: `help(WeatherClient)` lists the parameters `transport`, `max_retries` and `sleep`.

In [ ]:
import sys
from datetime import date

import httpx

sys.path.insert(0, "sessions/16-deployment-and-monitoring/workbooks")   # the folder of weather.py
from weather import APIError, WeatherClient

ONE_DAY = {"daily": {"time": ["2024-07-01"], "temperature_2m_mean": [17.6],
                     "precipitation_sum": [7.9], "sunshine_duration": [33378.58]}}

## Part 2 · Architecture in the cloud, monitoring and maintenance

### 2.1 · Draw the architecture from memory

*Paper · 10 min*

> Without looking at the lecture notebook, draw the architecture of the tariff application: front end, back end (the model API) and database, and the arrows between them with the protocol on each arrow (HTTPS, HTTP + JSON, SQL). Then write next to each part the AWS service that would run it in the cloud, and mark the one place where confidential requests could be stored.

*Source:* original exercise for this course; background: [AWS Well-Architected Framework](https://docs.aws.amazon.com/wellarchitected/latest/framework/welcome.html)

*Your answer:*



### 2.2 · PSI by hand

*Paper · 10 min*

> The shares of the issuing countries, grouped into three groups, in the training data (2017–2023) and in the requests of 2024:
>
> | Group | Training data | 2024 |
> |---|---|---|
> | Germany (DE) | 0.56 | 0.62 |
> | United Kingdom (GB) | 0.04 | 0.00 |
> | all other member states | 0.40 | 0.38 |
>
> Compute the population stability index. Is the shift stable, moderate or large by the usual rule of thumb? Which group contributes most?

*Source:* [Yurdakul (2018), *Statistical properties of population stability index*, PhD thesis, Western Michigan University](https://scholarworks.wmich.edu/dissertations/3208/)

*Your answer:*



### 2.3 · Country drift, one period later

*Code · 15 min*

> The lecture computed the PSI of the issuing country between the training sample and the requests of 2024. Compute it for the requests of 2025–2026 (the rest of `test.parquet`). Then compute it again for both periods with all GB decisions removed from the reference and from the current data. What do the four numbers say about the drift that is left once the United Kingdom is taken out?

In [ ]:
import pandas as pd

train = pd.read_parquet("case-study/data/train_sample.parquet")
test = pd.read_parquet("case-study/data/test.parquet")          # 2024-2026 requests, without labels
print(len(train), "training decisions;", len(test), "requests from", test["start_date"].min().date(),
      "to", test["start_date"].max().date())

### 2.4 · Feedback loops

*Class · 5 min*

> Customs officers start accepting the model's suggestions, more and more often without changing them. What happens to next year's training labels, and to the accuracy you measure on them?

*Source:* [Sculley et al. (2015), "Hidden technical debt in machine learning systems", NeurIPS](https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html)

*Your answer:*



### 2.5 · Results per country for the model card (*Stretch*)

*Code · 15 min*

> The model card should report results per factor. The lecture showed them per language. Train version 1 (word TF-IDF and linear SVM, as in the lecture) on the training sample, predict the 2024 requests and compute the accuracy per issuing country against the released 2024 labels, for the countries with at least 500 decisions. Which countries are served worst, and is the issuing country a separate factor from the language?

In [ ]:
FEEDBACK = Path(os.environ.get("FEEDBACK_PATH", "case-study/data/feedback_2024.csv"))
print("feedback file found" if FEEDBACK.exists() else f"{FEEDBACK} is missing: the lecturer releases it in Block 2")